In [0]:
from pyspark.sql import functions as F

landing_path = "/Volumes/nyc-yellow/default/landing"

files = [
    "yellow_tripdata_2023-01.parquet",
    "yellow_tripdata_2023-02.parquet",
    "yellow_tripdata_2023-03.parquet",
    "yellow_tripdata_2023-04.parquet",
    "yellow_tripdata_2023-05.parquet"
]

def read_and_standardize(file_name):
    df = spark.read.parquet(f"{landing_path}/{file_name}")

    return (
        df.select(
            F.col("VendorID").cast("int").alias("vendor_id"),
            F.col("passenger_count").cast("int").alias("passenger_count"),
            F.col("total_amount").cast("double").alias("total_amount"),
            F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_datetime"),
            F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_datetime")
        )
        .withColumn("source_file", F.lit(file_name))
    )

df_list = [read_and_standardize(file) for file in files]

df_landing = df_list[0]
for df in df_list[1:]:
    df_landing = df_landing.unionByName(df)

display(df_landing.limit(10))

vendor_id,passenger_count,total_amount,pickup_datetime,dropoff_datetime,source_file
2,1,14.3,2023-01-01T00:32:10.000Z,2023-01-01T00:40:36.000Z,yellow_tripdata_2023-01.parquet
2,1,16.9,2023-01-01T00:55:08.000Z,2023-01-01T01:01:27.000Z,yellow_tripdata_2023-01.parquet
2,1,34.9,2023-01-01T00:25:04.000Z,2023-01-01T00:37:49.000Z,yellow_tripdata_2023-01.parquet
1,0,20.85,2023-01-01T00:03:48.000Z,2023-01-01T00:13:25.000Z,yellow_tripdata_2023-01.parquet
2,1,19.68,2023-01-01T00:10:29.000Z,2023-01-01T00:21:19.000Z,yellow_tripdata_2023-01.parquet
2,1,27.8,2023-01-01T00:50:34.000Z,2023-01-01T01:02:52.000Z,yellow_tripdata_2023-01.parquet
2,1,20.52,2023-01-01T00:09:22.000Z,2023-01-01T00:19:49.000Z,yellow_tripdata_2023-01.parquet
2,1,64.44,2023-01-01T00:27:12.000Z,2023-01-01T00:49:56.000Z,yellow_tripdata_2023-01.parquet
2,1,28.38,2023-01-01T00:21:44.000Z,2023-01-01T00:36:40.000Z,yellow_tripdata_2023-01.parquet
2,1,19.9,2023-01-01T00:39:42.000Z,2023-01-01T00:50:36.000Z,yellow_tripdata_2023-01.parquet


In [0]:
from pyspark.sql import functions as F

# ============================================================
# CAMADA BRONZE
# Objetivo:
# Gravar os dados consolidados da Landing Zone em uma tabela
# gerenciada no Databricks, preservando a granularidade original.
# ============================================================

# Como o catálogo possui hífen no nome, usamos crase.
bronze_table = "`nyc-yellow`.default.bronze_taxi_trip"

# Gravação da tabela Bronze
(
    df_landing
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

print(f"Tabela Bronze criada com sucesso: {bronze_table}")

# ============================================================
# VALIDAÇÃO DA CAMADA BRONZE
# ============================================================

df_bronze = spark.table(bronze_table)

print(f"Quantidade de registros na Bronze: {df_bronze.count():,}")
print(f"Quantidade de colunas na Bronze: {len(df_bronze.columns)}")

display(df_bronze.limit(10))

df_bronze.printSchema()

Tabela Bronze criada com sucesso: `nyc-yellow`.default.bronze_taxi_trip
Quantidade de registros na Bronze: 16,186,386
Quantidade de colunas na Bronze: 6


vendor_id,passenger_count,total_amount,pickup_datetime,dropoff_datetime,source_file
2,1,11.1,2023-03-01T00:06:43.000Z,2023-03-01T00:16:43.000Z,yellow_tripdata_2023-03.parquet
2,2,76.49,2023-03-01T00:08:25.000Z,2023-03-01T00:39:30.000Z,yellow_tripdata_2023-03.parquet
1,0,28.05,2023-03-01T00:15:04.000Z,2023-03-01T00:29:26.000Z,yellow_tripdata_2023-03.parquet
1,1,24.7,2023-03-01T00:49:37.000Z,2023-03-01T01:01:05.000Z,yellow_tripdata_2023-03.parquet
2,1,14.64,2023-03-01T00:08:04.000Z,2023-03-01T00:11:06.000Z,yellow_tripdata_2023-03.parquet
1,1,18.0,2023-03-01T00:09:09.000Z,2023-03-01T00:17:34.000Z,yellow_tripdata_2023-03.parquet
1,1,20.5,2023-03-01T00:32:21.000Z,2023-03-01T00:42:08.000Z,yellow_tripdata_2023-03.parquet
1,1,15.7,2023-03-01T00:45:12.000Z,2023-03-01T00:52:37.000Z,yellow_tripdata_2023-03.parquet
1,1,40.4,2023-03-01T00:19:43.000Z,2023-03-01T00:39:37.000Z,yellow_tripdata_2023-03.parquet
2,1,22.2,2023-03-01T00:08:42.000Z,2023-03-01T00:18:45.000Z,yellow_tripdata_2023-03.parquet


root
 |-- vendor_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

# ============================================================
# CAMADA SILVER - AJUSTADA
# Objetivo:
# Aplicar regras de qualidade, padronização, enriquecimento
# e filtro de escopo temporal do case: Jan/2023 a Mai/2023.
# ============================================================

bronze_table = "`nyc-yellow`.default.bronze_taxi_trip"
silver_table = "`nyc-yellow`.default.silver_taxi_trip"

df_bronze = spark.table(bronze_table)

df_silver = (
    df_bronze
    .select(
        F.col("vendor_id").cast("int").alias("vendor_id"),
        F.col("passenger_count").cast("int").alias("passenger_count"),
        F.col("total_amount").cast("double").alias("total_amount"),
        F.col("pickup_datetime").cast("timestamp").alias("pickup_datetime"),
        F.col("dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),
        F.col("source_file")
    )

    # Remove registros com campos obrigatórios nulos
    .filter(F.col("vendor_id").isNotNull())
    .filter(F.col("passenger_count").isNotNull())
    .filter(F.col("total_amount").isNotNull())
    .filter(F.col("pickup_datetime").isNotNull())
    .filter(F.col("dropoff_datetime").isNotNull())

    # Mantém somente o período solicitado no case: Jan/2023 a Mai/2023
    .filter(
        (F.col("pickup_datetime") >= F.to_timestamp(F.lit("2023-01-01 00:00:00"))) &
        (F.col("pickup_datetime") <= F.to_timestamp(F.lit("2023-05-31 23:59:59")))
    )

    # Regras básicas de qualidade
    .filter(F.col("passenger_count") > 0)
    .filter(F.col("total_amount") > 0)
    .filter(F.col("dropoff_datetime") > F.col("pickup_datetime"))

    # Colunas derivadas para análise
    .withColumn("pickup_date", F.to_date("pickup_datetime"))
    .withColumn("pickup_month", F.date_format("pickup_datetime", "yyyy-MM"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
)

# Gravação da Silver ajustada
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Tabela Silver recriada com sucesso: {silver_table}")

# Validação
df_silver_validation = spark.table(silver_table)

print(f"Quantidade de registros na Bronze: {df_bronze.count():,}")
print(f"Quantidade de registros na Silver ajustada: {df_silver_validation.count():,}")
print(f"Quantidade de colunas na Silver ajustada: {len(df_silver_validation.columns)}")

display(df_silver_validation.limit(10))

df_silver_validation.printSchema()

Tabela Silver recriada com sucesso: `nyc-yellow`.default.silver_taxi_trip
Quantidade de registros na Bronze: 16,186,386
Quantidade de registros na Silver ajustada: 15,335,892
Quantidade de colunas na Silver ajustada: 9


vendor_id,passenger_count,total_amount,pickup_datetime,dropoff_datetime,source_file,pickup_date,pickup_month,pickup_hour
2,1,11.1,2023-03-01T00:06:43.000Z,2023-03-01T00:16:43.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
2,2,76.49,2023-03-01T00:08:25.000Z,2023-03-01T00:39:30.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
1,1,24.7,2023-03-01T00:49:37.000Z,2023-03-01T01:01:05.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
2,1,14.64,2023-03-01T00:08:04.000Z,2023-03-01T00:11:06.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
1,1,18.0,2023-03-01T00:09:09.000Z,2023-03-01T00:17:34.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
1,1,20.5,2023-03-01T00:32:21.000Z,2023-03-01T00:42:08.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
1,1,15.7,2023-03-01T00:45:12.000Z,2023-03-01T00:52:37.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
1,1,40.4,2023-03-01T00:19:43.000Z,2023-03-01T00:39:37.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
2,1,22.2,2023-03-01T00:08:42.000Z,2023-03-01T00:18:45.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0
2,1,15.3,2023-03-01T00:48:06.000Z,2023-03-01T00:57:15.000Z,yellow_tripdata_2023-03.parquet,2023-03-01,2023-03,0


root
 |-- vendor_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_month: string (nullable = true)
 |-- pickup_hour: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

# ============================================================
# CAMADA GOLD
# Objetivo:
# Disponibilizar a tabela final de consumo analítico,
# baseada na Silver já tratada e restrita ao escopo do case.
# ============================================================

silver_table = "`nyc-yellow`.default.silver_taxi_trip"
gold_table = "`nyc-yellow`.default.gold_taxi_trip"

df_silver = spark.table(silver_table)

df_gold = (
    df_silver
    .select(
        "vendor_id",
        "passenger_count",
        "total_amount",
        "pickup_datetime",
        "dropoff_datetime",
        "pickup_date",
        "pickup_month",
        "pickup_hour"
    )
)

# Gravação da Gold
(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("pickup_month")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"Tabela Gold recriada com sucesso: {gold_table}")

# Validação
df_gold_validation = spark.table(gold_table)

print(f"Quantidade de registros na Gold ajustada: {df_gold_validation.count():,}")
print(f"Quantidade de colunas na Gold ajustada: {len(df_gold_validation.columns)}")

display(df_gold_validation.limit(10))

df_gold_validation.printSchema()

Tabela Gold recriada com sucesso: `nyc-yellow`.default.gold_taxi_trip
Quantidade de registros na Gold ajustada: 15,335,892
Quantidade de colunas na Gold ajustada: 8


vendor_id,passenger_count,total_amount,pickup_datetime,dropoff_datetime,pickup_date,pickup_month,pickup_hour
2,1,11.1,2023-03-01T00:06:43.000Z,2023-03-01T00:16:43.000Z,2023-03-01,2023-03,0
2,2,76.49,2023-03-01T00:08:25.000Z,2023-03-01T00:39:30.000Z,2023-03-01,2023-03,0
1,1,24.7,2023-03-01T00:49:37.000Z,2023-03-01T01:01:05.000Z,2023-03-01,2023-03,0
2,1,14.64,2023-03-01T00:08:04.000Z,2023-03-01T00:11:06.000Z,2023-03-01,2023-03,0
1,1,18.0,2023-03-01T00:09:09.000Z,2023-03-01T00:17:34.000Z,2023-03-01,2023-03,0
1,1,20.5,2023-03-01T00:32:21.000Z,2023-03-01T00:42:08.000Z,2023-03-01,2023-03,0
1,1,15.7,2023-03-01T00:45:12.000Z,2023-03-01T00:52:37.000Z,2023-03-01,2023-03,0
1,1,40.4,2023-03-01T00:19:43.000Z,2023-03-01T00:39:37.000Z,2023-03-01,2023-03,0
2,1,22.2,2023-03-01T00:08:42.000Z,2023-03-01T00:18:45.000Z,2023-03-01,2023-03,0
2,1,15.3,2023-03-01T00:48:06.000Z,2023-03-01T00:57:15.000Z,2023-03-01,2023-03,0


root
 |-- vendor_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_month: string (nullable = true)
 |-- pickup_hour: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

df_gold = spark.table("`nyc-yellow`.default.gold_taxi_trip")

avg_total_amount_month = (
    df_gold
    .groupBy("pickup_month")
    .agg(
        F.round(
            F.avg("total_amount"),
            2
        ).alias("avg_total_amount")
    )
    .orderBy("pickup_month")
)

display(avg_total_amount_month)

pickup_month,avg_total_amount
2023-01,27.46
2023-02,27.37
2023-03,28.29
2023-04,28.78
2023-05,29.45


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

df_gold = spark.table("`nyc-yellow`.default.gold_taxi_trip")

avg_passengers_hour = (
    df_gold
    .filter(F.col("pickup_month") == "2023-05")
    .groupBy("pickup_hour")
    .agg(
        F.round(
            F.avg("passenger_count"),
            2
        ).alias("avg_passenger_count")
    )
    .orderBy("pickup_hour")
)

display(avg_passengers_hour)

pickup_hour,avg_passenger_count
0,1.43
1,1.44
2,1.46
3,1.45
4,1.41
5,1.28
6,1.26
7,1.28
8,1.3
9,1.31


Databricks visualization. Run in Databricks to view.